# FactoryShield-OT: Week 1 - Dataset Exploration & EDA

**Target:** HAIEnd 23.05 / HAI 23.05 Dataset  
**Industrial Environment:** Emerson Ovation DCS (Boiler Process P1)  
**Features:** 225 data points (35 SCADA I/O tags + 190 internal control logic edges)  
**Clean Training Samples:** 896,400  
**Attack Ratio:** ~4% in test sets

## Objectives
1. Understand the dataset structure and feature space
2. Analyze temporal continuity and sampling frequency  
3. Examine tag distributions and statistical properties
4. Identify missing values and data quality issues
5. Establish baseline understanding for preprocessing

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from datetime import datetime

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import project modules
from src.data.dataloader import HAIDataLoader
from src.data.preprocess import HAIPreprocessor

# Setup plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Dataset Loading & Initial Inspection

In [ ]:
# Define data paths
DATA_DIR = project_root / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Available raw files:")
for file in RAW_DIR.glob("*.csv"):
    print(f"  - {file.name} ({file.stat().st_size / 1024**2:.1f} MB)")

# Select a sample file for exploration
sample_file = RAW_DIR / "end-train1.csv"  # Adjust based on actual files
print(f"\nSelected file: {sample_file.name}")

In [ ]:
# Load the dataset
loader = HAIDataLoader(sample_file)
df = loader.df

# Basic information
print("Dataset Overview:")
print(f"Shape: {df.shape} (rows x columns)")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Time range: {df[loader.timestamp_col].min()} to {df[loader.timestamp_col].max()}")
print(f"Duration: {(df[loader.timestamp_col].max() - df[loader.timestamp_col].min())}")
print(f"Sampling interval: ~{(df[loader.timestamp_col].diff().mean().total_seconds()):.2f} seconds")

In [ ]:
# Display summary
summary = loader.get_summary()
print("\nDataset Summary:")
for key, value in summary.items():
    if key != 'time_range':
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value[0]} to {value[1]}")

## 2. Temporal Analysis

In [ ]:
# Analyze temporal characteristics
time_series = df[loader.timestamp_col]
time_diffs = time_series.diff().dt.total_seconds().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series overview
axes[0, 0].plot(time_series, range(len(time_series)), 'b-', alpha=0.7)
axes[0, 0].set_xlabel('Timestamp')
axes[0, 0].set_ylabel('Sample Index')
axes[0, 0].set_title('Temporal Progression')
axes[0, 0].tick_params(axis='x', rotation=45)

# Sampling interval distribution
axes[0, 1].hist(time_diffs, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(time_diffs.mean(), color='red', linestyle='--', label=f'Mean: {time_diffs.mean():.2f}s')
axes[0, 1].axvline(time_diffs.median(), color='green', linestyle='--', label=f'Median: {time_diffs.median():.2f}s')
axes[0, 1].set_xlabel('Sampling Interval (seconds)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Sampling Interval Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Missing timestamps analysis
expected_interval = time_diffs.mode()[0] if not time_diffs.mode().empty else time_diffs.median()
gap_threshold = expected_interval * 2
gaps = time_diffs[time_diffs > gap_threshold]

if len(gaps) > 0:
    axes[1, 0].bar(range(len(gaps)), gaps.values, edgecolor='black', alpha=0.7)
    axes[1, 0].axhline(gap_threshold, color='red', linestyle='--', label=f'Gap threshold: {gap_threshold:.1f}s')
    axes[1, 0].set_xlabel('Gap Index')
axes[1, 0].set_ylabel('Gap Duration (seconds)')
axes[1, 0].set_title(f'Temporal Gaps (> {gap_threshold:.1f}s): {len(gaps)} found')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Time of day analysis
df['hour'] = time_series.dt.hour
hourly_counts = df['hour'].value_counts().sort_index()
axes[1, 1].bar(hourly_counts.index, hourly_counts.values, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Sample Count')
axes[1, 1].set_title('Data Distribution by Hour')
axes[1, 1].set_xticks(range(0, 24, 2))
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTemporal Analysis:")
print(f"  Average sampling interval: {time_diffs.mean():.2f} seconds")
print(f"  Median sampling interval: {time_diffs.median():.2f} seconds")
print(f"  Sampling interval std: {time_diffs.std():.2f} seconds")
print(f"  Temporal gaps (> {gap_threshold:.1f}s): {len(gaps)}")
if len(gaps) > 0:
    print(f"    Largest gap: {gaps.max():.1f} seconds")
    print(f"    Total missing time: {gaps.sum():.1f} seconds ({gaps.sum()/3600:.2f} hours)")

## 3. Feature Analysis & Sensor Distributions

In [ ]:
# Get sensor columns
sensor_cols = loader.get_sensor_columns()
print(f"Number of sensor columns: {len(sensor_cols)}")
print(f"First 10 sensors: {sensor_cols[:10]}")

# Select a subset of sensors for detailed analysis
sample_sensors = sensor_cols[:8]  # First 8 sensors

# Statistical summary
sensor_stats = df[sample_sensors].describe().T
sensor_stats['range'] = sensor_stats['max'] - sensor_stats['min']
sensor_stats['cv'] = sensor_stats['std'] / sensor_stats['mean']  # Coefficient of variation

print("\nSensor Statistics (sample):")
print(sensor_stats[['mean', 'std', 'min', 'max', 'range', 'cv']].round(4))

In [ ]:
# Visualize sensor distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, sensor in enumerate(sample_sensors):
    if idx < len(axes):
        data = df[sensor].dropna()
        
        # Histogram with KDE
        axes[idx].hist(data, bins=50, density=True, alpha=0.6, color='skyblue', edgecolor='black')
        
        # Add KDE
        from scipy.stats import gaussian_kde
        if len(data) > 1:
            kde = gaussian_kde(data)
            x_range = np.linspace(data.min(), data.max(), 100)
            axes[idx].plot(x_range, kde(x_range), 'r-', linewidth=2, alpha=0.8)
        
        axes[idx].set_title(f'{sensor}\nMean: {data.mean():.2f}, Std: {data.std():.2f}')
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Density')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Time series visualization for selected sensors
fig, axes = plt.subplots(4, 2, figsize=(16, 12))
axes = axes.flatten()

# Plot first 1000 samples for clarity
plot_samples = min(1000, len(df))

for idx, sensor in enumerate(sample_sensors):
    if idx < len(axes):
        axes[idx].plot(df[loader.timestamp_col].iloc[:plot_samples], 
                      df[sensor].iloc[:plot_samples], 
                      'b-', alpha=0.7, linewidth=1)
        axes[idx].set_title(sensor)
        axes[idx].set_xlabel('Time')
        axes[idx].set_ylabel('Value')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Missing Values Analysis

In [ ]:
# Analyze missing values
missing_analysis = loader.check_missing_values()

# Filter columns with missing values
columns_with_missing = missing_analysis[missing_analysis['n_missing'] > 0]

print(f"Columns with missing values: {len(columns_with_missing)} out of {len(df.columns)}")

if len(columns_with_missing) > 0:
    print("\nTop 10 columns with most missing values:")
    print(columns_with_missing.sort_values('pct_missing', ascending=False).head(10))
    
    # Visualize missing values pattern
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Missing values percentage
    top_missing = columns_with_missing.sort_values('pct_missing', ascending=False).head(15)
    axes[0].barh(range(len(top_missing)), top_missing['pct_missing'].values)
    axes[0].set_yticks(range(len(top_missing)))
    axes[0].set_yticklabels(top_missing.index)
    axes[0].set_xlabel('Missing Percentage (%)')
    axes[0].set_title('Top 15 Columns by Missing Percentage')
    axes[0].grid(True, alpha=0.3)
    
    # Missing values heatmap (first 50 columns for clarity)
    missing_matrix = df.isna().astype(int).iloc[:500, :50]  # First 500 rows, first 50 columns
    axes[1].imshow(missing_matrix.T, aspect='auto', cmap='Reds', interpolation='nearest')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Column Index')
    axes[1].set_title('Missing Values Pattern (First 500 samples, 50 columns)')
    axes[1].set_yticks(range(0, 50, 5))
    axes[1].set_yticklabels(range(0, 50, 5))
    
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values detected in the dataset!")

## 5. Attack Label Analysis

In [ ]:
# Analyze attack labels
if loader.label_cols:
    print(f"Attack label columns found: {loader.label_cols}")
    
    # Get global labels
    global_labels = loader.get_global_labels()
    
    # Label distribution
    unique_labels, counts = np.unique(global_labels, return_counts=True)
    label_dist = pd.DataFrame({
        'Label': ['Normal (0)', 'Anomaly (1)'],
        'Count': counts,
        'Percentage': counts / len(global_labels) * 100
    })
    
    print("\nGlobal Label Distribution:")
    print(label_dist.to_string(index=False))
    
    # Visualize label distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart
    axes[0].pie(label_dist['Count'], labels=label_dist['Label'], autopct='%1.1f%%', 
                colors=['lightgreen', 'lightcoral'], startangle=90)
    axes[0].set_title('Attack Label Distribution')
    
    # Time series of labels
    axes[1].plot(df[loader.timestamp_col].iloc[:plot_samples], 
                global_labels[:plot_samples], 
                'r-', alpha=0.7, linewidth=1)
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Attack Label (0/1)')
    axes[1].set_title('Attack Labels Over Time')
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(['Normal', 'Attack'])
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Analyze attack segments
    attack_indices = np.where(global_labels == 1)[0]
    if len(attack_indices) > 0:
        # Find contiguous attack segments
        segments = []
        current_segment = [attack_indices[0]]
        for i in range(1, len(attack_indices)):
            if attack_indices[i] == attack_indices[i-1] + 1:
                current_segment.append(attack_indices[i])
            else:
                segments.append(current_segment)
                current_segment = [attack_indices[i]]
        segments.append(current_segment)
        
        print(f"\nAttack Segment Analysis:")
        print(f"  Total attack segments: {len(segments)}")
        print(f"  Total attack samples: {len(attack_indices)}")
        
        segment_lengths = [len(seg) for seg in segments]
        print(f"  Min segment length: {min(segment_lengths)} samples")
        print(f"  Max segment length: {max(segment_lengths)} samples")
        print(f"  Mean segment length: {np.mean(segment_lengths):.1f} samples")
        print(f"  Median segment length: {np.median(segment_lengths):.1f} samples")
        
        # Segment length distribution
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(segment_lengths, bins=20, edgecolor='black', alpha=0.7)
        ax.set_xlabel('Segment Length (samples)')
        ax.set_ylabel('Frequency')
        ax.set_title('Attack Segment Length Distribution')
        ax.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No attack samples found in this dataset.")
else:
    print("No attack label columns found in this dataset.")

## 6. Correlation Analysis

In [ ]:
# Analyze correlations between sensors
if len(sample_sensors) >= 4:
    # Select a subset for correlation analysis
    corr_sensors = sample_sensors[:8]
    corr_matrix = df[corr_sensors].corr()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Heatmap
    im = axes[0].imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    axes[0].set_xticks(range(len(corr_sensors)))
    axes[0].set_yticks(range(len(corr_sensors)))
    axes[0].set_xticklabels(corr_sensors, rotation=45, ha='right')
    axes[0].set_yticklabels(corr_sensors)
    axes[0].set_title('Sensor Correlation Matrix')
    plt.colorbar(im, ax=axes[0])
    
    # Add correlation values
    for i in range(len(corr_sensors)):
        for j in range(len(corr_sensors)):
            axes[0].text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', 
                        ha='center', va='center', color='black', fontsize=9)
    
    # Distribution of correlation coefficients
    corr_values = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
    axes[1].hist(corr_values, bins=20, edgecolor='black', alpha=0.7)
    axes[1].axvline(corr_values.mean(), color='red', linestyle='--', 
                    label=f'Mean: {corr_values.mean():.2f}')
    axes[1].axvline(corr_values.std(), color='green', linestyle='--', 
                    label=f'Std: {corr_values.std():.2f}')
    axes[1].set_xlabel('Correlation Coefficient')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Correlation Coefficients')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nCorrelation Analysis:")
    print(f"  Mean absolute correlation: {np.abs(corr_values).mean():.3f}")
    print(f"  Strong correlations (>0.8): {np.sum(np.abs(corr_values) > 0.8)}/{len(corr_values)}")
    print(f"  Weak correlations (<0.2): {np.sum(np.abs(corr_values) < 0.2)}/{len(corr_values)}")

## 7. Data Quality Assessment

In [ ]:
# Comprehensive data quality assessment
print("Data Quality Assessment Report")
print("=" * 40)

# 1. Completeness
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
completeness_score = 100 * (1 - missing_cells / total_cells)
print(f"1. Completeness: {completeness_score:.2f}% ({missing_cells} missing values)")

# 2. Temporal consistency
time_consistency = 100 * (time_diffs.std() / time_diffs.mean()) if time_diffs.mean() > 0 else 100
print(f"2. Temporal Consistency: CV = {time_consistency:.2f}%")
print(f"   Expected interval: {expected_interval:.2f}s, Std: {time_diffs.std():.2f}s")

# 3. Sensor value ranges
sensor_ranges = df[sensor_cols].describe().loc[['min', 'max']]
unusual_ranges = ((sensor_ranges.loc['min'] < -1e6) | (sensor_ranges.loc['max'] > 1e6)).sum()
print(f"3. Value Ranges: {unusual_ranges}/{len(sensor_cols)} sensors with unusual ranges")

# 4. Duplicate timestamps
duplicate_times = df[loader.timestamp_col].duplicated().sum()
duplicate_score = 100 * (1 - duplicate_times / len(df))
print(f"4. Duplicate Timestamps: {duplicate_score:.2f}% ({duplicate_times} duplicates)")

# 5. Monotonic time
monotonic = df[loader.timestamp_col].is_monotonic_increasing
print(f"5. Monotonic Time: {'✅ Yes' if monotonic else '❌ No'}")

# Overall quality score
quality_score = (
    completeness_score * 0.3 +
    max(0, 100 - time_consistency) * 0.2 +
    (100 - (unusual_ranges / len(sensor_cols) * 100)) * 0.2 +
    duplicate_score * 0.2 +
    (100 if monotonic else 0) * 0.1
)

print("\nOverall Data Quality Score:")
if quality_score >= 90:
    print(f"  🎯 EXCELLENT: {quality_score:.1f}/100")
elif quality_score >= 75:
    print(f"  👍 GOOD: {quality_score:.1f}/100")
elif quality_score >= 60:
    print(f"  ⚠️ FAIR: {quality_score:.1f}/100")
else:
    print(f"  ❌ POOR: {quality_score:.1f}/100")

print("\nRecommendations:")
if missing_cells > 0:
    print("  - Implement imputation strategy for missing values")
if time_consistency > 10:
    print("  - Consider resampling to regular intervals")
if unusual_ranges > 0:
    print("  - Check sensor calibration and outlier detection")
if duplicate_times > 0:
    print("  - Remove duplicate timestamps")
if not monotonic:
    print("  - Sort data by timestamp")

## 8. Key Findings & Next Steps

In [ ]:
# Summary of key findings
print("Key Findings from EDA:")
print("=" * 40)

findings = [
    f"Dataset: {df.shape[0]:,} samples × {df.shape[1]} features",
    f"Temporal coverage: {summary['time_range'][0]} to {summary['time_range'][1]}",
    f"Sampling rate: ~{time_diffs.mean():.2f} seconds",
    f"Sensor columns: {len(sensor_cols)}",
    f"Missing values: {missing_cells} cells ({completeness_score:.1f}% complete)",
]

if loader.label_cols:
    global_labels = loader.get_global_labels()
    anomaly_percentage = np.sum(global_labels) / len(global_labels) * 100
    findings.append(f"Anomaly rate: {anomaly_percentage:.2f}%")

for finding in findings:
    print(f"• {finding}")

print("\nNext Steps for Week 2:")
print("1. Implement MinMaxScaler fitting on normal data only")
print("2. Develop PyTorch DataLoader with sliding windows")
print("3. Train baseline Isolation Forest model")
print("4. Establish evaluation metrics pipeline")

print("\n✅ EDA Completed Successfully!")